# QA Prompt Tuning

所有要改的东西都在下面这一个 cell 里。改完直接跑下面的「运行」cell 看效果。

In [2]:
import os, sys, json, re
sys.path.append(os.path.dirname(os.getcwd()))

from openai import OpenAI
from config import API_BASE, API_KEY, MODEL

client = OpenAI(base_url=API_BASE, api_key=API_KEY, timeout=600.0)

---
## 配置区（改这里）

In [ ]:
# ============================================================
# 一、目标书籍（调 prompt 时一次只跑一本）
# ============================================================
BOOKS = ["Oliver Twist.cleaned.txt"]  # 调 prompt 用第一本，批量生产时跑全部


# ============================================================
# 二、总结提示词（控制 LLM 如何写 section summary）
# ============================================================
SUMMARY_PROMPT = """You are a literary analyst. For the given book section, write a concise summary covering:
- Key events and plot developments
- Character introductions, interactions, and relationships
- How each character is referred to — full names, titles, nicknames, surname-only references, pseudonyms

Pay special attention to name variations: if a character is called by different names in this section (e.g., "Mr. Boldwood" and "William Boldwood"), note this explicitly — it will be crucial for alias extraction later.

Output ONLY the summary text. Do NOT extract triples or add any other sections."""


# ============================================================
# 三、出题提示词（控制 LLM 生成什么类型的 QA）
# ============================================================
QA_PROMPT = """You are given section summaries of one or more books.
Based on the summaries, generate 3-5 high-quality question-answer pairs about the book content.
Each QA pair should test deep understanding of characters, relationships, events, and plot.

Requirements:
- Questions should be diverse: some about characters, some about events, some about relationships.
- Answers should be accurate and cite specific details from the summaries.
- CRITICAL: For each factual claim in each answer, cite the source section number(s) in brackets:
  [3] or [5][6][7]. Every distinct fact must have a citation.
  Do NOT use range notation like [5-7]; write [5][6][7] instead.
- Avoid yes/no questions; prefer open-ended questions requiring reasoning.
- Questions must NOT contain section numbers (e.g., "in Section X").

Output format — a JSON array of objects:
[
    {
        "book": "<book title>",
        "question": "<question>",
        "answer": "<answer with [N] citations>"
    },
    ...
]

Return ONLY the JSON array, no other text."""


# ============================================================
# 四、其他参数（和 question_types.py 对齐）
# ============================================================
TEMPERATURE = 0.3
NUM_QUESTIONS = "3-5"

---
## 运行（不要改这里）

In [ ]:
# ════════════════════════════════════════════════════════════════
# 运行（不要改这里）
# ════════════════════════════════════════════════════════════════

import importlib
import pipeline as _pl
importlib.reload(_pl)

from pipeline import process_books
from config import WORKERS

result = process_books(
    selected_names=BOOKS,
    question=None,
    question_type=None,
    override_prompts={
        "label": "notebook调试",
        "summary_prompt": SUMMARY_PROMPT,
        "qa_prompt": QA_PROMPT,
    },
)

print(result["answer_with_evidence"])

---
## 满意后抄回 question_types.py

把上面「配置区」的 `SUMMARY_PROMPT` 和 `QA_PROMPT` 内容复制到 `question_types.py` 的对应条目。